# Email Spam Classification: Traditional Machine Learning vs. Deep Learning
# Phân loại email spam: Học máy truyền thống và học sâu

**Vietnamese / Tiếng Việt**

Notebook này xây dựng mô hình phân loại nhị phân để dự đoán một email là **spam** hay **ham (không spam)** dựa trên nội dung văn bản. Các mô hình được so sánh gồm:

- Mô hình truyền thống: Multinomial Naive Bayes, Logistic Regression, Linear SVM với đặc trưng TF-IDF.
- Mô hình học sâu: Bidirectional LSTM và Bidirectional GRU với lớp Embedding.

**English**

This notebook builds binary text classifiers that predict whether an email is **spam** or **ham (not spam)** from its textual content. It compares:

- Traditional models: Multinomial Naive Bayes, Logistic Regression, and Linear SVM with TF-IDF features.
- Deep learning models: Bidirectional LSTM and Bidirectional GRU with an Embedding layer.

**Colab setup:** Open this notebook in Google Colab, choose `Runtime > Change runtime type > T4 GPU`, then choose `Runtime > Run all`.

## 1. Problem statement / Phát biểu bài toán

**VI:** Đây là bài toán phân loại nhị phân có giám sát. Đầu vào là nội dung email, đầu ra là nhãn `0 = ham` hoặc `1 = spam`. Ngoài Accuracy, ta cần quan tâm Precision, Recall, F1-score và ROC-AUC. Với lọc spam, Recall cao giúp hạn chế bỏ sót spam; Precision cao giúp tránh đưa nhầm email hợp lệ vào thư mục spam.

**EN:** This is a supervised binary classification problem. The input is an email's text and the output is `0 = ham` or `1 = spam`. In addition to Accuracy, we evaluate Precision, Recall, F1-score, and ROC-AUC. For spam filtering, high Recall reduces missed spam, while high Precision reduces legitimate emails incorrectly moved to the spam folder.

## 2. Dataset / Bộ dữ liệu

**VI:** Notebook sử dụng bộ dữ liệu công khai [`SetFit/enron_spam`](https://huggingface.co/datasets/SetFit/enron_spam) trên Hugging Face. Bộ dữ liệu gồm email Enron đã được gán nhãn spam/ham. Trường `text` chứa nội dung dùng để huấn luyện và trường `label` chứa nhãn.

**EN:** This notebook uses the public [`SetFit/enron_spam`](https://huggingface.co/datasets/SetFit/enron_spam) dataset hosted on Hugging Face. It contains labeled Enron spam and ham emails. The `text` field is used as input and `label` is the target.

> Note: `FAST_RUN=True` uses a stratified subset so the notebook can run within a free Colab session. Set it to `False` for the full cleaned dataset.

In [ ]:
# Reproducibility and runtime configuration
import gc
import html
import random
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
FAST_RUN = True
MAX_TOTAL_SAMPLES = 18_000
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("FAST_RUN:", FAST_RUN)

In [ ]:
DATA_URL = "https://huggingface.co/datasets/SetFit/enron_spam/resolve/main/enron_spam_data.csv"

raw_df = pd.read_csv(DATA_URL)
df = raw_df[["text", "label"]].copy()
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(int)
df = df[df["text"].str.len() > 0]
df = df.drop_duplicates(subset="text").reset_index(drop=True)

if FAST_RUN and len(df) > MAX_TOTAL_SAMPLES:
    df, _ = train_test_split(
        df,
        train_size=MAX_TOTAL_SAMPLES,
        stratify=df["label"],
        random_state=SEED,
    )
    df = df.reset_index(drop=True)

label_names = {0: "ham", 1: "spam"}
df["label_name"] = df["label"].map(label_names)
df["text_length"] = df["text"].str.len()

print("Dataset shape after cleaning and optional sampling:", df.shape)
display(df.head(3))

## 2.1 Vietnamese labeled seed data / Dữ liệu seed tiếng Việt đã gán nhãn

**VI:** Phần này bổ sung email tiếng Việt được gán nhãn thủ công: `0 = ham`, `1 = spam`. Tập seed thủ công được chia vào train/validation/test. Các biến thể sinh thêm chỉ được thêm vào **tập train sau khi chia dữ liệu**, vì vậy chúng không làm rò rỉ dữ liệu vào validation hoặc test. Đây là dữ liệu khởi đầu phục vụ bài tập, không thay thế một bộ dữ liệu email tiếng Việt thực tế có quy mô lớn.

**EN:** This section adds manually labeled Vietnamese emails: `0 = ham`, `1 = spam`. Manual seed samples are split into train/validation/test. Additional generated variants are added **only to the training split after splitting**, so they do not leak into validation or test. This is starter data for the assignment, not a replacement for a large real-world Vietnamese email dataset.

In [ ]:
INCLUDE_VIETNAMESE_SEED = True
VI_AUGMENT_PER_CLASS = 300

# Manually labeled Vietnamese seed emails: 1 = spam, 0 = ham.
vi_spam_labeled = [
    ("prize_fee", "Chúc mừng bạn đã trúng thưởng xe máy. Vui lòng chuyển khoản phí nhận thưởng để hoàn tất thủ tục."),
    ("prize_otp", "Bạn là khách hàng may mắn nhận giải thưởng 50 triệu đồng. Hãy gửi mã OTP để xác nhận nhận quà."),
    ("gift_link", "Thông báo nhận quà miễn phí từ chương trình tri ân. Bấm vào liên kết ngay hôm nay để đăng ký."),
    ("bank_phishing", "Tài khoản ngân hàng của bạn sắp bị khóa. Vui lòng truy cập đường link và xác minh tài khoản ngay lập tức."),
    ("bank_phishing", "Phát hiện giao dịch bất thường. Hãy đăng nhập ngay theo liên kết bên dưới và cung cấp mã OTP để hủy giao dịch."),
    ("bank_phishing", "Tài khoản của quý khách đã tạm khóa. Bấm vào link để cập nhật mật khẩu và thông tin cá nhân."),
    ("impersonation", "Chúng tôi là cơ quan công an đang điều tra vụ án rửa tiền. Yêu cầu bạn chuyển khoản để xác minh tài sản."),
    ("impersonation", "Bạn có liên quan đến vụ án. Hãy cung cấp căn cước công dân và chuyển tiền vào tài khoản an toàn để kiểm tra."),
    ("impersonation", "Tòa án thông báo bạn có lệnh triệu tập khẩn cấp. Liên hệ Telegram ngay để được hướng dẫn nộp phí xử lý."),
    ("job_scam", "Tuyển cộng tác viên online việc nhẹ lương cao. Chỉ cần nạp tiền làm nhiệm vụ để nhận hoa hồng mỗi ngày."),
    ("job_scam", "Làm việc tại nhà thu nhập 2 triệu mỗi ngày. Kết bạn Zalo và đặt cọc để bắt đầu công việc."),
    ("job_scam", "Cơ hội kiếm tiền nhanh không cần kinh nghiệm. Ứng trước 500 nghìn đồng để kích hoạt tài khoản nhiệm vụ."),
    ("investment", "Cơ hội đầu tư lợi nhuận cao cam kết lãi 30 phần trăm mỗi tháng. Nạp tiền ngay để nhận ưu đãi."),
    ("investment", "Tham gia nhóm đầu tư crypto nội bộ, lợi nhuận đảm bảo. Chuyển khoản hôm nay để giữ suất VIP."),
    ("investment", "Gói đầu tư sinh lời không rủi ro. Liên hệ Telegram và nạp tiền để chuyên gia giao dịch giúp bạn."),
    ("delivery_fee", "Đơn hàng của bạn đang bị giữ tại kho. Vui lòng thanh toán phí vận chuyển qua đường link để nhận hàng."),
    ("delivery_fee", "Bưu kiện chưa thể giao do thiếu phí. Hãy quét liên kết và chuyển khoản 25 nghìn đồng ngay hôm nay."),
    ("loan_scam", "Hồ sơ vay của bạn đã được duyệt. Nộp phí bảo hiểm trước để giải ngân khoản vay trong ngày."),
    ("loan_scam", "Hỗ trợ vay nhanh không cần chứng minh thu nhập. Vui lòng đóng phí hồ sơ để nhận tiền ngay."),
    ("lottery", "Bạn đã trúng giải đặc biệt. Cung cấp số căn cước và mã OTP để nhận tiền thưởng."),
    ("account_upgrade", "Nâng cấp tài khoản định danh điện tử khẩn cấp. Bấm vào link và nhập thông tin cá nhân để tránh bị khóa."),
    ("tax_refund", "Bạn được hoàn thuế 3 triệu đồng. Truy cập liên kết để khai báo tài khoản ngân hàng và nhận tiền."),
    ("charity_scam", "Kêu gọi ủng hộ khẩn cấp. Hãy chuyển khoản ngay vào tài khoản cá nhân để hỗ trợ bệnh nhân."),
    ("unaccented_phishing", "KHAN CAP: tai khoan cua ban bi khoa. Vui long bam vao link de xac minh tai khoan va gui ma OTP."),
]

vi_ham_labeled = [
    ("work", "Chào cả nhóm, vui lòng xem lại tiến độ dự án và cập nhật phần việc trước cuộc họp sáng mai."),
    ("work", "Mình gửi biên bản cuộc họp chiều nay. Nhờ mọi người phản hồi nếu có nội dung cần bổ sung."),
    ("work", "Anh gửi tài liệu thiết kế mới. Em kiểm tra giúp anh các hạng mục còn thiếu trước thứ Sáu nhé."),
    ("work", "Phòng nhân sự thông báo lịch đào tạo nội bộ vào 9 giờ sáng thứ Hai tại phòng họp tầng 3."),
    ("work", "Khách hàng đã xác nhận yêu cầu. Đội kỹ thuật vui lòng cập nhật kế hoạch triển khai trong tuần này."),
    ("education", "Thầy gửi lịch học bù môn xử lý ngôn ngữ tự nhiên. Các em xem tài liệu trước khi đến lớp."),
    ("education", "Thông báo nộp bài tập nhóm trước 23 giờ Chủ nhật trên hệ thống học tập của trường."),
    ("education", "Mời bạn tham dự hội thảo khoa học tại hội trường A. Vui lòng xác nhận số người tham gia."),
    ("bank_notice", "Bạn đã chuyển khoản 500 nghìn đồng thành công. Không cung cấp mã OTP cho bất kỳ ai."),
    ("bank_notice", "Ngân hàng thông báo biến động số dư. Nếu không thực hiện giao dịch, vui lòng gọi tổng đài chính thức."),
    ("bank_notice", "Thẻ của quý khách đã thanh toán tại siêu thị. Đây là email thông báo tự động, không yêu cầu phản hồi."),
    ("bank_notice", "Mã OTP chỉ dùng để xác thực giao dịch của bạn. Tuyệt đối không gửi mã này cho người khác."),
    ("delivery", "Đơn hàng của bạn đã được giao thành công. Cảm ơn bạn đã mua sắm tại cửa hàng."),
    ("delivery", "Nhân viên giao hàng sẽ liên hệ trong hôm nay. Bạn có thể kiểm tra trạng thái trong ứng dụng chính thức."),
    ("delivery", "Hóa đơn điện tử của đơn hàng được đính kèm trong email này để bạn tiện lưu trữ."),
    ("personal", "Cuối tuần này gia đình mình họp mặt lúc 18 giờ. Bạn thu xếp đến sớm giúp mình nhé."),
    ("personal", "Mình đã gửi ảnh chuyến đi qua thư mục dùng chung. Khi rảnh bạn xem và chọn ảnh cần in nhé."),
    ("personal", "Cảm ơn bạn đã hỗ trợ hôm qua. Mình sẽ hoàn thiện báo cáo và gửi lại trong chiều nay."),
    ("service", "Lịch bảo trì hệ thống dự kiến bắt đầu lúc 22 giờ và kết thúc lúc 23 giờ 30 phút tối nay."),
    ("service", "Yêu cầu hỗ trợ của bạn đã được ghi nhận. Bộ phận kỹ thuật sẽ phản hồi trong vòng hai ngày làm việc."),
    ("service", "Gói dịch vụ của bạn sẽ hết hạn vào cuối tháng. Bạn có thể gia hạn trong ứng dụng nếu còn nhu cầu."),
    ("newsletter", "Bản tin tháng này tổng hợp các hoạt động nổi bật và lịch sự kiện sắp diễn ra của câu lạc bộ."),
    ("newsletter", "Thư viện thông báo đã bổ sung giáo trình mới. Sinh viên có thể đến mượn trong giờ hành chính."),
    ("meeting", "Mời bạn tham gia buổi trao đổi trực tuyến vào 14 giờ chiều mai. Đường dẫn họp nằm trong lịch công việc."),
]

vi_seed_df = pd.DataFrame(
    [(text, 1, category) for category, text in vi_spam_labeled]
    + [(text, 0, category) for category, text in vi_ham_labeled],
    columns=["text", "label", "category"],
)
vi_seed_df["source"] = "manual_vi_seed"
vi_seed_df["language"] = "vi"

df["source"] = "enron_spam"
df["language"] = "en"
if INCLUDE_VIETNAMESE_SEED:
    df = pd.concat([df, vi_seed_df], ignore_index=True)
    df = df.drop_duplicates(subset="text").reset_index(drop=True)

df["label_name"] = df["label"].map(label_names)
df["text_length"] = df["text"].str.len()

print("Vietnamese manual seed samples:", len(vi_seed_df))
print("Combined dataset shape:", df.shape)
display(vi_seed_df.groupby(["label", "category"]).size().rename("samples").reset_index())

def build_vietnamese_train_augmentation(samples_per_class=VI_AUGMENT_PER_CLASS):
    """Create labeled Vietnamese variants for training only; never add them to validation or test."""
    rng = random.Random(SEED)
    spam_parts = [
        [
            "THÔNG BÁO KHẨN: tài khoản của bạn đang bị tạm khóa.",
            "Chúc mừng bạn đã trúng thưởng phần quà giá trị cao.",
            "Cơ hội đầu tư lợi nhuận cao dành riêng cho bạn.",
            "Đơn hàng của bạn đang bị giữ tại kho do thiếu phí.",
            "Hồ sơ vay nhanh của bạn đã được duyệt trong hôm nay.",
            "Tuyen cong tac vien online viec nhe luong cao.",
            "Cơ quan điều tra thông báo bạn liên quan đến vụ án rửa tiền.",
            "Bạn đủ điều kiện nhận khoản hoàn thuế đặc biệt.",
        ],
        [
            "Vui lòng bấm vào liên kết để xác minh tài khoản.",
            "Hãy chuyển khoản phí xử lý để nhận thưởng.",
            "Vui lòng cung cấp mã OTP để hoàn tất thủ tục.",
            "Kết bạn Zalo và nạp tiền để bắt đầu nhiệm vụ.",
            "Liên hệ Telegram để được hướng dẫn chuyển tiền.",
            "Hãy nhập thông tin căn cước công dân qua đường link.",
            "Vui long gui ma OTP va xac minh tai khoan.",
            "Nộp phí bảo hiểm trước để giải ngân khoản vay.",
        ],
        [
            "Thực hiện ngay hôm nay để tránh bị khóa.",
            "Ưu đãi sẽ hết hạn trong vòng 24 giờ.",
            "Không chia sẻ thông báo này với bất kỳ ai.",
            "Bạn cần xử lý ngay lập tức.",
            "Chỉ còn một suất cuối cùng.",
            "Nếu chậm trễ hồ sơ sẽ bị hủy.",
            "Hãy làm theo hướng dẫn để nhận tiền ngay.",
            "Phản hồi ngay để tránh phát sinh phí bổ sung.",
        ],
    ]
    ham_parts = [
        [
            "Chào bạn, mình gửi thông tin cuộc họp tuần này.",
            "Phòng nhân sự thông báo lịch đào tạo nội bộ.",
            "Ngân hàng gửi thông báo giao dịch thành công.",
            "Nhà trường gửi lịch học và thời hạn nộp bài.",
            "Cửa hàng xác nhận đơn hàng đã được giao.",
            "Bộ phận kỹ thuật đã tiếp nhận yêu cầu hỗ trợ.",
            "Mình gửi bạn tài liệu cập nhật của dự án.",
            "Thư viện gửi bản tin hoạt động trong tháng.",
        ],
        [
            "Vui lòng xem lại nội dung và phản hồi khi thuận tiện.",
            "Bạn có thể kiểm tra chi tiết trong ứng dụng chính thức.",
            "Không cung cấp mã OTP cho bất kỳ ai.",
            "Nhờ bạn cập nhật tiến độ trước buổi trao đổi.",
            "Tài liệu được đính kèm để bạn tiện lưu trữ.",
            "Nếu cần hỗ trợ, vui lòng gọi tổng đài chính thức.",
            "Mọi người đọc trước tài liệu để chuẩn bị thảo luận.",
            "Cảm ơn bạn đã sử dụng dịch vụ của chúng tôi.",
        ],
        [
            "Trân trọng.",
            "Cảm ơn bạn.",
            "Hẹn gặp bạn trong buổi họp.",
            "Chúc bạn một ngày làm việc hiệu quả.",
            "Đây là email thông báo tự động.",
            "Thân mến.",
            "Nhờ bạn xác nhận đã nhận được email.",
            "Chúc bạn cuối tuần vui vẻ.",
        ],
    ]

    rows = []
    for label, part_groups in [(1, spam_parts), (0, ham_parts)]:
        for _ in range(samples_per_class):
            rows.append(
                {
                    "text": " ".join(rng.choice(parts) for parts in part_groups),
                    "label": label,
                    "category": "synthetic_train_only",
                    "source": "synthetic_vi_train",
                    "language": "vi",
                }
            )
    return pd.DataFrame(rows).drop_duplicates(subset="text").reset_index(drop=True)

## 3. Exploratory data analysis / Phân tích dữ liệu khám phá

**VI:** Ta kiểm tra phân bố nhãn và độ dài email. Nếu hai lớp mất cân bằng mạnh, Accuracy có thể gây hiểu nhầm; khi đó F1-score và ROC-AUC quan trọng hơn.

**EN:** We inspect label distribution and email lengths. If the classes are highly imbalanced, Accuracy can be misleading; F1-score and ROC-AUC then become more informative.

In [ ]:
class_summary = (
    df["label_name"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)
class_summary["percentage"] = (100 * class_summary["count"] / len(df)).round(2)
display(class_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="label_name", order=["ham", "spam"], ax=axes[0])
axes[0].set_title("Class distribution / Phân bố nhãn")
axes[0].set_xlabel("Class / Nhãn")

sns.histplot(
    data=df,
    x="text_length",
    hue="label_name",
    bins=60,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_xlim(0, df["text_length"].quantile(0.98))
axes[1].set_title("Email length up to 98th percentile / Độ dài email")
plt.tight_layout()
plt.show()

## 4. Text preprocessing and split / Tiền xử lý và chia dữ liệu

**VI:** Ta thay URL, địa chỉ email và số bằng token chung, chuyển chữ thường và chuẩn hóa khoảng trắng. Sau đó dữ liệu được chia theo tỷ lệ `70% train - 15% validation - 15% test` bằng stratified split để duy trì tỷ lệ spam/ham. Các biến thể tiếng Việt sinh thêm chỉ được nối vào train sau bước chia. Tập test chỉ dùng để đánh giá cuối cùng.

**EN:** URLs, email addresses, and numbers are replaced with generic tokens, text is lowercased, and whitespace is normalized. Data is then split into `70% train - 15% validation - 15% test` using stratification to preserve the spam/ham ratio. Additional Vietnamese variants are appended only to train after splitting. The test set is used only for final evaluation.

In [ ]:
def normalize_text(text):
    """Apply light normalization while preserving useful email vocabulary."""
    text = html.unescape(str(text))
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b[\w.%-]+@[\w.-]+\.[a-z]{2,}\b", " EMAIL ", text)
    text = re.sub(r"\d+", " NUMBER ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].map(normalize_text)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
)

if INCLUDE_VIETNAMESE_SEED:
    vi_aug_train_df = build_vietnamese_train_augmentation()
    vi_aug_train_df["clean_text"] = vi_aug_train_df["text"].map(normalize_text)
    vi_aug_train_df = vi_aug_train_df.drop_duplicates(subset="clean_text").reset_index(drop=True)
    train_df = pd.concat([train_df, vi_aug_train_df], ignore_index=True)
    print("Vietnamese generated samples added to train only:", len(vi_aug_train_df))

X_train, y_train = train_df["clean_text"], train_df["label"]
X_val, y_val = val_df["clean_text"], val_df["label"]
X_test, y_test = test_df["clean_text"], test_df["label"]

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "samples": [len(train_df), len(val_df), len(test_df)],
        "spam_ratio": [y_train.mean(), y_val.mean(), y_test.mean()],
    }
)
display(split_summary)

vietnamese_split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "vietnamese_samples": [
            int((train_df["language"] == "vi").sum()),
            int((val_df["language"] == "vi").sum()),
            int((test_df["language"] == "vi").sum()),
        ],
        "generated_train_only": [
            int((train_df["source"] == "synthetic_vi_train").sum()),
            0,
            0,
        ],
    }
)
display(vietnamese_split_summary)

## 5. Evaluation helpers / Hàm đánh giá

**VI:** Mọi mô hình được đánh giá trên cùng tập test. Điều này giúp so sánh hợp lý giữa mô hình truyền thống và học sâu.

**EN:** Every model is evaluated on the same test set. This makes the comparison between traditional and deep learning models consistent.

In [ ]:
results = []
test_predictions = {}
test_probabilities = {}

def add_result(model_name, family, y_true, y_pred, y_score, train_seconds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    result = {
        "model": model_name,
        "family": family,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc_score(y_true, y_score),
        "train_seconds": train_seconds,
    }
    results.append(result)
    test_predictions[model_name] = np.asarray(y_pred)
    test_probabilities[model_name] = np.asarray(y_score)
    return result

def show_report(model_name):
    print(f"Classification report: {model_name}")
    print(
        classification_report(
            y_test,
            test_predictions[model_name],
            target_names=["ham", "spam"],
            digits=4,
        )
    )

def plot_confusion(model_names):
    fig, axes = plt.subplots(1, len(model_names), figsize=(5 * len(model_names), 4))
    axes = np.atleast_1d(axes)
    for ax, model_name in zip(axes, model_names):
        cm = confusion_matrix(y_test, test_predictions[model_name])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
        ax.set_title(model_name)
        ax.set_xlabel("Predicted label / Nhãn dự đoán")
        ax.set_ylabel("True label / Nhãn thật")
        ax.set_xticklabels(["ham", "spam"])
        ax.set_yticklabels(["ham", "spam"], rotation=0)
    plt.tight_layout()
    plt.show()

## 6. Traditional models with TF-IDF / Mô hình truyền thống với TF-IDF

**VI:** TF-IDF biểu diễn email thành vector dựa trên mức độ quan trọng của từ và cụm hai từ. Các baseline này thường huấn luyện nhanh, dễ giải thích và rất cạnh tranh trong bài toán phân loại văn bản.

**EN:** TF-IDF transforms emails into vectors based on the importance of words and bigrams. These baselines are usually fast to train, interpretable, and highly competitive for text classification.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=30_000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

start = time.perf_counter()
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)
tfidf_seconds = time.perf_counter() - start

print("TF-IDF train matrix shape:", X_train_tfidf.shape)
print(f"TF-IDF fit and transform time: {tfidf_seconds:.2f} seconds")

In [ ]:
traditional_models = {
    "MultinomialNB": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(
        max_iter=1_000,
        class_weight="balanced",
        random_state=SEED,
    ),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=SEED),
}

for model_name, model in traditional_models.items():
    start = time.perf_counter()
    model.fit(X_train_tfidf, y_train)
    train_seconds = time.perf_counter() - start
    y_pred = model.predict(X_test_tfidf)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_tfidf)[:, 1]
    else:
        y_score = model.decision_function(X_test_tfidf)
    metrics = add_result(
        model_name,
        "Traditional ML",
        y_test,
        y_pred,
        y_score,
        train_seconds,
    )
    print(model_name, {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

In [ ]:
traditional_results = pd.DataFrame(results).sort_values("f1", ascending=False)
display(traditional_results.style.format({
    "accuracy": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "train_seconds": "{:.2f}",
}))

best_traditional = traditional_results.iloc[0]["model"]
show_report(best_traditional)
plot_confusion([best_traditional])

## 7. Deep learning preparation / Chuẩn bị cho học sâu

**VI:** `TextVectorization` chuyển văn bản thành chuỗi chỉ số token có độ dài cố định. LSTM và GRU học biểu diễn embedding và quan hệ theo thứ tự từ. Ta dùng Early Stopping để giảm overfitting và phục hồi trọng số tốt nhất theo validation loss.

**EN:** `TextVectorization` converts text into fixed-length token-index sequences. LSTM and GRU learn embeddings and sequential relationships between words. Early Stopping limits overfitting and restores the weights with the best validation loss.

In [ ]:
MAX_TOKENS = 25_000
SEQUENCE_LENGTH = 300
EMBEDDING_DIM = 128
RNN_UNITS = 64
BATCH_SIZE = 64
EPOCHS = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train.to_numpy()).batch(256))

AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(texts, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (texts.to_numpy(dtype=str), labels.to_numpy(dtype=np.float32))
    )
    if training:
        dataset = dataset.shuffle(len(texts), seed=SEED)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))
print("Example token sequence shape:", vectorizer(X_train.iloc[:2]).shape)

In [ ]:
def build_rnn_model(cell_type):
    inputs = tf.keras.Input(shape=(), dtype=tf.string, name="email_text")
    x = vectorizer(inputs)
    x = tf.keras.layers.Embedding(
        input_dim=MAX_TOKENS,
        output_dim=EMBEDDING_DIM,
        mask_zero=True,
        name="embedding",
    )(x)
    if cell_type == "LSTM":
        x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(RNN_UNITS))(x)
    elif cell_type == "GRU":
        x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(RNN_UNITS))(x)
    else:
        raise ValueError("cell_type must be 'LSTM' or 'GRU'")
    x = tf.keras.layers.Dropout(0.40)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="spam_probability")(x)

    model = tf.keras.Model(inputs, outputs, name=f"Bi{cell_type}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-5,
    ),
]

## 8. Train LSTM and GRU / Huấn luyện LSTM và GRU

**VI:** Hai kiến trúc sử dụng cùng cách tiền xử lý và siêu tham số chính để việc so sánh có ý nghĩa. Thời gian huấn luyện có thể khác nhau tùy GPU Colab.

**EN:** Both architectures use the same preprocessing and main hyperparameters to make the comparison meaningful. Training time may vary depending on the Colab GPU.

In [ ]:
deep_models = {}
histories = {}

for cell_type in ["LSTM", "GRU"]:
    model_name = f"Bi{cell_type}"
    print("\n" + "=" * 70)
    print("Training", model_name)
    model = build_rnn_model(cell_type)
    model.summary()

    start = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.perf_counter() - start

    y_score = model.predict(test_ds, verbose=0).reshape(-1)
    y_pred = (y_score >= 0.5).astype(int)
    metrics = add_result(
        model_name,
        "Deep Learning",
        y_test,
        y_pred,
        y_score,
        train_seconds,
    )
    print(model_name, {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

    deep_models[model_name] = model
    histories[model_name] = history.history
    gc.collect()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for model_name, history in histories.items():
    axes[0].plot(history["loss"], marker="o", label=f"{model_name} train")
    axes[0].plot(history["val_loss"], marker="o", linestyle="--", label=f"{model_name} val")
    axes[1].plot(history["auc"], marker="o", label=f"{model_name} train")
    axes[1].plot(history["val_auc"], marker="o", linestyle="--", label=f"{model_name} val")

axes[0].set_title("Loss by epoch / Loss theo epoch")
axes[1].set_title("AUC by epoch / AUC theo epoch")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 9. Final comparison / So sánh cuối cùng

**VI:** Bảng dưới đây là kết quả chính để đưa vào báo cáo. Không nên kết luận rằng học sâu luôn tốt hơn: với dữ liệu văn bản vừa phải, TF-IDF và mô hình tuyến tính thường đạt chất lượng cao với thời gian huấn luyện ngắn hơn đáng kể. LSTM/GRU có khả năng học quan hệ theo trình tự nhưng cần nhiều tài nguyên hơn.

**EN:** The following table is the main result for the report. Deep learning should not automatically be assumed to perform better: for moderate-sized text datasets, TF-IDF with linear models often achieves strong quality with much shorter training time. LSTM/GRU can learn sequential relationships but require more compute.

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1", ascending=False).reset_index(drop=True)
display(results_df.style.format({
    "accuracy": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "train_seconds": "{:.2f}",
}).background_gradient(subset=["accuracy", "precision", "recall", "f1", "roc_auc"], cmap="Greens"))

plt.figure(figsize=(10, 4))
sns.barplot(data=results_df, x="model", y="f1", hue="family", dodge=False)
plt.ylim(max(0, results_df["f1"].min() - 0.05), 1.0)
plt.title("F1-score comparison / So sánh F1-score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["model"]
best_deep_name = results_df[results_df["family"] == "Deep Learning"].iloc[0]["model"]
best_traditional_name = results_df[results_df["family"] == "Traditional ML"].iloc[0]["model"]

print("Best overall model / Mô hình tốt nhất:", best_model_name)
print("Best traditional model / Mô hình truyền thống tốt nhất:", best_traditional_name)
print("Best deep model / Mô hình học sâu tốt nhất:", best_deep_name)
plot_confusion([best_traditional_name, best_deep_name])

## 9.1 Vietnamese manual test subset / Đánh giá riêng trên email tiếng Việt thủ công

**VI:** Bảng dưới đây chỉ đánh giá các email tiếng Việt thủ công xuất hiện trong `test_df`. Dữ liệu sinh thêm không được đưa vào test. Vì số mẫu seed còn nhỏ, kết quả này chỉ dùng để tham khảo và minh họa, không phải benchmark đủ mạnh cho triển khai thực tế.

**EN:** The table below evaluates only manually labeled Vietnamese emails that appear in `test_df`. Generated variants are never added to test. Because the seed set is small, these results are directional and illustrative rather than a production-grade benchmark.

In [ ]:
vi_test_mask = test_df["language"].eq("vi").to_numpy()
print("Vietnamese manual test samples:", int(vi_test_mask.sum()))

if vi_test_mask.any():
    y_test_vi = y_test.to_numpy()[vi_test_mask]
    vi_results = []
    for model_name in results_df["model"]:
        y_pred_vi = test_predictions[model_name][vi_test_mask]
        y_score_vi = test_probabilities[model_name][vi_test_mask]
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test_vi, y_pred_vi, average="binary", zero_division=0
        )
        vi_results.append(
            {
                "model": model_name,
                "accuracy": accuracy_score(y_test_vi, y_pred_vi),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc_auc": roc_auc_score(y_test_vi, y_score_vi)
                if len(np.unique(y_test_vi)) == 2
                else np.nan,
            }
        )
    vi_results_df = pd.DataFrame(vi_results).sort_values("f1", ascending=False)
    display(vi_results_df.style.format({
        "accuracy": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "roc_auc": "{:.4f}",
    }))
else:
    print("No Vietnamese manual samples were assigned to test_df.")

In [ ]:
best_row = results_df.iloc[0]
traditional_row = results_df[results_df["family"] == "Traditional ML"].iloc[0]
deep_row = results_df[results_df["family"] == "Deep Learning"].iloc[0]

print("KẾT LUẬN TỰ ĐỘNG / AUTOMATIC CONCLUSION")
print("-" * 70)
print(
    f"VI: Mô hình tốt nhất theo F1-score là {best_row['model']} "
    f"(F1={best_row['f1']:.4f}, Accuracy={best_row['accuracy']:.4f}, "
    f"ROC-AUC={best_row['roc_auc']:.4f})."
)
print(
    f"VI: Baseline truyền thống tốt nhất là {traditional_row['model']} "
    f"(F1={traditional_row['f1']:.4f}, thời gian train={traditional_row['train_seconds']:.2f}s); "
    f"mô hình học sâu tốt nhất là {deep_row['model']} "
    f"(F1={deep_row['f1']:.4f}, thời gian train={deep_row['train_seconds']:.2f}s)."
)
print()
print(
    f"EN: The best model by F1-score is {best_row['model']} "
    f"(F1={best_row['f1']:.4f}, Accuracy={best_row['accuracy']:.4f}, "
    f"ROC-AUC={best_row['roc_auc']:.4f})."
)
print(
    f"EN: The best traditional baseline is {traditional_row['model']} "
    f"(F1={traditional_row['f1']:.4f}, training time={traditional_row['train_seconds']:.2f}s); "
    f"the best deep learning model is {deep_row['model']} "
    f"(F1={deep_row['f1']:.4f}, training time={deep_row['train_seconds']:.2f}s)."
)

## 10. Error analysis / Phân tích lỗi

**VI:** Xem một số false positive và false negative của mô hình tốt nhất. False positive là email hợp lệ bị chặn nhầm; false negative là spam bị bỏ sót. Đây là cơ sở để điều chỉnh threshold hoặc bổ sung dữ liệu.

**EN:** Inspect false positives and false negatives from the best model. False positives are legitimate emails incorrectly blocked; false negatives are missed spam. This analysis can guide threshold tuning or data collection.

In [ ]:
error_df = test_df[["text", "label"]].copy().reset_index(drop=True)
error_df["prediction"] = test_predictions[best_model_name]
error_df["score"] = test_probabilities[best_model_name]
error_df["error_type"] = np.select(
    [
        (error_df["label"] == 0) & (error_df["prediction"] == 1),
        (error_df["label"] == 1) & (error_df["prediction"] == 0),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

for error_type in ["false_positive", "false_negative"]:
    print("\n", error_type.upper())
    examples = error_df[error_df["error_type"] == error_type].copy()
    examples["text"] = examples["text"].str.slice(0, 400)
    display(examples[["label", "prediction", "score", "text"]].head(5))

## 11. Predict new emails / Dự đoán email mới

**VI:** Hàm dưới đây dùng mô hình có F1-score cao nhất để dự đoán văn bản mới. Với Linear SVM, `score` là decision score; với các mô hình khác, `score` là xác suất spam.

**EN:** The function below uses the model with the highest F1-score to classify new text. For Linear SVM, `score` is a decision score; for other models, it is the spam probability.

In [ ]:
def predict_email(email_text, model_name=None):
    model_name = model_name or best_model_name
    clean_email = normalize_text(email_text)

    if model_name in traditional_models:
        features = tfidf.transform([clean_email])
        model = traditional_models[model_name]
        prediction = int(model.predict(features)[0])
        if hasattr(model, "predict_proba"):
            score = float(model.predict_proba(features)[0, 1])
        else:
            score = float(model.decision_function(features)[0])
    else:
        model = deep_models[model_name]
        score = float(model.predict(np.array([clean_email]), verbose=0)[0, 0])
        prediction = int(score >= 0.5)

    return {
        "model": model_name,
        "prediction": label_names[prediction],
        "score": round(score, 4),
    }

sample_emails = [
    "Congratulations! You won a free vacation. Click here now to claim your prize.",
    "Hi team, please review the attached meeting notes before tomorrow's project call.",
    "THÔNG BÁO KHẨN: Bạn đã trúng thưởng 50 triệu đồng. Vui lòng chuyển khoản phí nhận thưởng và gửi mã OTP ngay hôm nay.",
    "Chào cả nhóm, nhờ mọi người cập nhật tiến độ dự án trước cuộc họp sáng mai.",
]

for email in sample_emails:
    print("Email:", email)
    print("Prediction:", predict_email(email))
    print()

## 12. Export results / Xuất kết quả

**VI:** File CSV có thể tải về từ thanh Files của Colab và đưa vào báo cáo.

**EN:** The CSV file can be downloaded from Colab's Files sidebar and included in the report.

In [ ]:
results_df.to_csv("model_comparison.csv", index=False)
print("Saved: model_comparison.csv")

# Optional in Google Colab:
# from google.colab import files
# files.download("model_comparison.csv")

## 13. Discussion for the report / Thảo luận cho báo cáo

**Vietnamese / Tiếng Việt**

1. **Mô hình truyền thống:** TF-IDF kết hợp Naive Bayes, Logistic Regression hoặc Linear SVM có ưu điểm là huấn luyện nhanh, ít tốn tài nguyên và là baseline mạnh cho phân loại văn bản.
2. **Mô hình học sâu:** LSTM và GRU có thể học thông tin theo thứ tự token. GRU thường có ít tham số hơn LSTM, còn LSTM dùng nhiều cổng hơn để kiểm soát luồng thông tin. Hiệu quả thực tế cần được xác nhận bằng bảng kết quả.
3. **Đánh đổi:** Nếu baseline tuyến tính đạt F1-score tương đương hoặc cao hơn, baseline là lựa chọn thực tế hơn cho hệ thống đơn giản. Nếu RNN cải thiện đáng kể chất lượng, chi phí huấn luyện và suy luận cao hơn có thể chấp nhận được.
4. **Hạn chế:** Dữ liệu Enron chủ yếu là tiếng Anh và có thể không đại diện cho email hiện đại hoặc email tiếng Việt. Mô hình triển khai thực tế cần dữ liệu mới, kiểm tra drift, điều chỉnh threshold và đánh giá false positive cẩn thận.
5. **Mở rộng:** Có thể fine-tune BERT/DistilBERT, bổ sung metadata của email, xử lý class imbalance, thử threshold khác `0.5`, và đánh giá cross-validation cho baseline.

**English**

1. **Traditional models:** TF-IDF combined with Naive Bayes, Logistic Regression, or Linear SVM trains quickly, consumes limited resources, and provides strong baselines for text classification.
2. **Deep learning models:** LSTM and GRU can learn token-order information. GRU typically has fewer parameters than LSTM, while LSTM uses more gates to control information flow. Actual performance must be verified using the results table.
3. **Trade-off:** If a linear baseline achieves a similar or higher F1-score, it is the more practical choice for a simple production system. If an RNN meaningfully improves quality, the higher training and inference costs may be justified.
4. **Limitations:** Enron data is mostly English and may not represent modern or Vietnamese emails. A production model requires recent data, drift monitoring, threshold tuning, and careful false-positive evaluation.
5. **Extensions:** Fine-tune BERT/DistilBERT, add email metadata, handle class imbalance, test thresholds other than `0.5`, and evaluate baselines using cross-validation.

## 14. Optional BERT extension / Mở rộng BERT tùy chọn

**VI:** Đề bài đã được đáp ứng bằng LSTM và GRU. Nếu cần mở rộng, có thể fine-tune DistilBERT hoặc BERT bằng Hugging Face Transformers. Transformer thường mạnh hơn trong việc tận dụng ngữ cảnh nhưng cần GPU, thời gian huấn luyện và bộ nhớ lớn hơn. Để giữ notebook chính ổn định trên Colab miễn phí, phần fine-tune BERT không được chạy mặc định.

**EN:** The assignment is already satisfied using LSTM and GRU. As an extension, DistilBERT or BERT can be fine-tuned with Hugging Face Transformers. Transformers often capture context more effectively but require more GPU memory and training time. To keep the main notebook reliable on free Colab sessions, BERT fine-tuning is not enabled by default.